# 05 — Near-duplicate leakage audit (MinHash LSH + exact Jaccard)

MinHash LSH is a candidate generator, not an answer, so every candidate pair it returns
is re-checked with an exact Jaccard computation over the full shingle sets. The threshold
is swept 0.5–0.9 under two shingle definitions (word 5-gram, character 8-gram) rather than
reported at a single convenient point.

- Input: `data/cleaned/*.json`, `results/predictions/preds_*.csv`
- Output: `results_r2/table_A3_minhash_sweep.csv`, `table_A4_neardup_free_eval.csv`
- Runtime: ~4 min, CPU
- Reported in: paper Section VI-C

## 1. Config

In [ ]:
# CONFIG
REPO_DIR    = '.'
RESULTS_DIR = './results_r2'

NUM_PERM   = 128
THRESHOLDS = [0.5, 0.6, 0.7, 0.8, 0.9]   # 0.8 is the reviewer's request
WORD_K     = 5    # word-shingle size
CHAR_K     = 8    # character-shingle size

IN_COLAB = False
try:
    import google.colab  # noqa
    IN_COLAB = True
except ImportError:
    pass
if IN_COLAB:
    from google.colab import drive; drive.mount('/content/drive')
    REPO_DIR    = '/content/drive/MyDrive/kazakh-ai-text-detection'
    RESULTS_DIR = '/content/drive/MyDrive/kazakh-ai-text-detection/results_r2'

import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'datasketch',
                'scikit-learn', 'pandas'], check=False)

# Auto-load the shared paths written by notebook 00_setup_and_check.
# Run notebook 00 once and you never edit a path in any notebook again.
# If r2_config.json is absent, the values set above are used unchanged.
try:
    import json as _json
    from pathlib import Path as _Path
    _cfg_path = _Path('/content/drive/MyDrive/r2_config.json')
    if _cfg_path.exists():
        _cfg = _json.load(open(_cfg_path))
        REPO_DIR = _cfg['REPO_DIR']
        RESULTS_DIR = _cfg.get('RESULTS_DIR', RESULTS_DIR)
        print('Paths loaded from r2_config.json')
        print('  REPO_DIR   =', REPO_DIR)
        print('  RESULTS_DIR=', RESULTS_DIR)
    else:
        print('r2_config.json not found -- using the paths set above. '
              'Run 00_setup_and_check.ipynb to generate it.')
except Exception as _e:
    print('Could not load r2_config.json (%s: %s) -- using the paths set above.'
          % (type(_e).__name__, _e))


## 2. Load all splits

In [ ]:
import json, re
from pathlib import Path
from collections import Counter
import numpy as np, pandas as pd
from datasketch import MinHash, MinHashLSH
from sklearn.metrics import f1_score

REPO = Path(REPO_DIR); OUT = Path(RESULTS_DIR); OUT.mkdir(parents=True, exist_ok=True)
ID2LABEL = {0: 'Human', 1: 'AI-Generated', 2: 'AI-Obfuscated'}

def load_split(n):
    d = pd.DataFrame(json.load(open(REPO / f'data/cleaned/{n}_cleaned.json', encoding='utf-8')))
    d['split'] = n; d['text'] = d['text'].astype(str); d['label'] = d['label'].astype(int)
    return d

train, dev, test = load_split('train'), load_split('dev'), load_split('test')
allsp = pd.concat([train, dev, test], ignore_index=True)
split_of, label_of = allsp['split'].to_numpy(), allsp['label'].to_numpy()
n_test = len(test); test_offset = len(train) + len(dev)
print('Corpus:', len(allsp), '| train/dev/test =', len(train), len(dev), n_test)


## 3. MinHash LSH sweep with exact Jaccard verification

In [ ]:
def word_shingles(t, k=WORD_K):
    toks = re.findall(r'\w+', t.lower(), flags=re.UNICODE)
    if not toks:
        return set()
    return {' '.join(toks[i:i + k]) for i in range(max(len(toks) - k + 1, 1))}

def char_shingles(t, k=CHAR_K):
    s = re.sub(r'\s+', ' ', t.lower())
    if not s:
        return set()
    return {s[i:i + k] for i in range(max(len(s) - k + 1, 1))}

def audit(shingle_fn, label):
    # LSH proposes candidates; every candidate is then verified with exact Jaccard.
    cache = [shingle_fn(t) for t in allsp['text']]
    sigs = []
    for sh in cache:
        m = MinHash(num_perm=NUM_PERM)
        for s in sh:
            m.update(s.encode('utf8'))
        sigs.append(m)

    rows, detail = [], {}
    for th in THRESHOLDS:
        lsh = MinHashLSH(threshold=th, num_perm=NUM_PERM)
        for i, m in enumerate(sigs):
            lsh.insert(str(i), m)

        candidates = set()
        for i, m in enumerate(sigs):
            for j in lsh.query(m):
                j = int(j)
                if j > i:
                    candidates.add((i, j))

        verified = []
        for i, j in candidates:                      # EXACT verification
            a, b = cache[i], cache[j]
            if not a or not b:
                continue
            jac = len(a & b) / len(a | b)
            if jac >= th:
                verified.append((i, j, jac))

        cross = Counter('/'.join(sorted((split_of[i], split_of[j]))) for i, j, _ in verified)
        contaminated = {a for i, j, _ in verified for a, b in ((i, j), (j, i))
                        if split_of[a] == 'test' and split_of[b] in ('train', 'dev')}
        rows.append({
            'shingle': label, 'threshold': th,
            'n_candidate_pairs': len(candidates), 'n_verified_pairs': len(verified),
            'within_train': cross.get('train/train', 0),
            'within_dev':   cross.get('dev/dev', 0),
            'within_test':  cross.get('test/test', 0),
            'train_dev':    cross.get('dev/train', 0),
            'train_test':   cross.get('test/train', 0),
            'dev_test':     cross.get('dev/test', 0),
            'n_test_docs_contaminated': len(contaminated),
            'pct_test_contaminated': round(100 * len(contaminated) / n_test, 2),
            'label_conflicting_pairs': sum(1 for i, j, _ in verified if label_of[i] != label_of[j]),
        })
        detail[th] = (verified, contaminated)
        print(rows[-1], flush=True)
    return rows, detail

rows_w, detail_w = audit(word_shingles, f'word {WORD_K}-gram')
rows_c, detail_c = audit(char_shingles, f'char {CHAR_K}-gram')

sweep = pd.DataFrame(rows_w + rows_c)
sweep.to_csv(OUT / 'table_A3_minhash_sweep.csv', index=False)
print('\n' + sweep.to_string(index=False))


## 4. Contamination summary

In [ ]:
# ---- Headline number for the rebuttal -------------------------------
at08 = sweep[sweep.threshold == 0.8]
cross_cols = ['train_dev', 'train_test', 'dev_test']
total_cross_08 = int(at08[cross_cols].to_numpy().sum())
total_cross_any = int(sweep[cross_cols].to_numpy().sum())

print(f'Cross-split near-duplicate pairs at Jaccard >= 0.8 : {total_cross_08}')
print(f'Cross-split near-duplicate pairs at ANY threshold  : {total_cross_any}')
print()
if total_cross_any == 0:
    print('=> No documents require pruning. Report this, plus the sweep, and delete the')
    print('   "fuzzy deduplication was not applied" sentence from Section III-C.')
else:
    print('=> Contamination found. The next cell re-scores every model on a clean test subset,')
    print('   and you must report BOTH the original and the cleaned figures.')


## 5. Re-score on a near-duplicate-free test set

In [ ]:
# ---- If (and only if) contamination exists: re-score on a clean subset ----
MODEL_FILES = {
    'mDeBERTa-v3 (base)': 'preds_mDeBERTa-v3_base.csv',
    'XLM-R (base)':       'preds_XLM-R_base.csv',
    'mBERT (cased)':      'preds_mBERT_cased.csv',
    'DistilmBERT':        'preds_DistilmBERT.csv',
    'TF-IDF + LogReg':    'preds_TF-IDF_+_LogReg.csv',
}
y_test = test['label'].to_numpy()

contaminated_global = set()
for detail in (detail_w, detail_c):
    for th, (ver, contam) in detail.items():
        contaminated_global |= contam
contaminated_local = sorted(i - test_offset for i in contaminated_global)
clean_idx = np.array([i for i in range(n_test) if i not in set(contaminated_local)])

rows = []
for m, f in MODEL_FILES.items():
    p = pd.read_csv(REPO / 'results/predictions' / f)
    assert (p['y_true'].to_numpy() == y_test).all(), f'{m}: predictions not aligned'
    yp = p['y_pred'].to_numpy()
    full  = f1_score(y_test, yp, average='macro')
    clean = f1_score(y_test[clean_idx], yp[clean_idx], average='macro')
    rows.append({'model': m, 'n_test': n_test, 'n_clean': len(clean_idx),
                 'n_removed': n_test - len(clean_idx),
                 'macro_f1_full': round(full, 4),
                 'macro_f1_no_neardup': round(clean, 4),
                 'delta': round(clean - full, 4)})

res = pd.DataFrame(rows)
res.to_csv(OUT / 'table_A4_neardup_free_eval.csv', index=False)
print(res.to_string(index=False))
